In [66]:
from ortools.sat.python import cp_model
import pandas as pd
import math
import ast

In [78]:
df = pd.read_csv("../data/verification_results.csv")

In [79]:
def data_cleanser(df):
    
    df = pd.read_csv("../data/verification_results.csv")
    try:
        df["rule_results"] = df["rule_results"].apply(ast.literal_eval)
        df_veri_candidates = df[df["verified"] == True][["record_id", "rule_results"]].copy()

    except Exception as e:
        print(f"Error processing rule_results: {e}")
        return None
    
    return df_veri_candidates, len(df)

In [80]:
df_verified, total_records = data_cleanser(df)
df_verified

,record_id,rule_results
0,record_1,"{'rule_1': ['b', 'c'], 'rule_2': ['b', 'c', 'f..."
1,record_10,"{'rule_1': ['a', 'd'], 'rule_2': ['d', 'f']}"
2,record_100,"{'rule_1': ['a', 'b', 'd', 'e', 'f'], 'rule_2'..."
3,record_1000,"{'rule_1': ['a'], 'rule_2': ['c']}"
4,record_101,"{'rule_1': ['g'], 'rule_2': ['c', 'd', 'e', 'h']}"
...,...,...
494,record_543,"{'rule_1': ['a', 'b'], 'rule_2': ['a', 'b']}"
495,record_544,"{'rule_1': ['a', 'b'], 'rule_2': ['a', 'b']}"
496,record_545,"{'rule_1': ['a'], 'rule_2': ['a', 'b', 'd']}"
497,record_546,"{'rule_1': ['b'], 'rule_2': ['d']}"


In [88]:
from ortools.sat.python import cp_model
import math


def ds_selection_optimizer(df_verified, source_cost, min_verified_rate=1, time_limit_sec=60):
    model = cp_model.CpModel()

    sources = sorted(source_cost.keys())

    total_records = len(df_verified)

    required_verified = math.ceil(total_records * min_verified_rate)

    x = {
        ds: model.NewBoolVar(f"x_{ds}")
        for ds in sources
    }

    z = {
        rc: model.NewBoolVar(f"z_{rc}")
        for rc in df_verified["record_id"]
    }

    y = {}

    for _, row in df_verified.iterrows():
        rc = row["record_id"]
        record_rules = row["rule_results"]
        rules = list(record_rules.keys())

        for ru in rules:
            ds_candidates = record_rules[ru]

            missing_sources = set(ds_candidates) - set(sources)
            if missing_sources:
                print(
                    f"Warning: data sources {missing_sources} "
                    f"for record {rc}, rule {ru} are not in source_cost."
                )

            valid_ds_candidates = [
                ds for ds in ds_candidates
                if ds in sources
            ]

            rule_assignment = []

            for ds in valid_ds_candidates:
                var = model.NewBoolVar(f"y_{rc}_{ru}_{ds}")
                y[(rc, ru, ds)] = var
                rule_assignment.append(var)

                # Cannot use data source unless globally selected
                model.Add(var <= x[ds])

            # If record is verified, this rule needs exactly one data source.
            # If record is not verified, this rule gets zero data sources.
            model.Add(sum(rule_assignment) == z[rc])

        # Same data source cannot be reused across rules for the same record
        for ds in sources:
            assignments_with_ds = [
                y[(rc, ru, ds)]
                for ru in rules
                if (rc, ru, ds) in y
            ]

            if assignments_with_ds:
                model.Add(sum(assignments_with_ds) <= 1)

    model.Add(sum(z[rc] for rc in z) >= required_verified)

    model.Minimize(
        sum(x[ds] * int(source_cost[ds]) for ds in sources)
    )

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit_sec

    status = solver.Solve(model)

    if status not in [cp_model.OPTIMAL, cp_model.FEASIBLE]:
        return {"status": "NO SOLUTION"}

    selected_sources = [
        ds for ds in sources
        if solver.Value(x[ds]) == 1
    ]

    verified_records = [
        rc for rc in z
        if solver.Value(z[rc]) == 1
    ]

    unverified_records = [
        rc for rc in z
        if solver.Value(z[rc]) == 0
    ]

    rule_assignments = {}

    for rc in verified_records:
        rule_assignments[rc] = {}

        rc_info = df_verified[df_verified["record_id"] == rc]
        record_rules = rc_info["rule_results"].values[0]

        for rule in record_rules.keys():
            rule_assignments[rc][rule] = []

            for ds in record_rules[rule]:
                if (rc, rule, ds) in y and solver.Value(y[(rc, rule, ds)]) == 1:
                    rule_assignments[rc][rule].append(ds)

    cost_per_record = sum(source_cost[ds] for ds in selected_sources)
    total_cost = cost_per_record * total_records

    return {
        "status": solver.StatusName(status),
        "min_verify_rate": min_verified_rate,
        "required_verified": required_verified,
        "actual_verified": len(verified_records),
        "actual_verify_rate": len(verified_records) / total_records,
        "selected_sources": selected_sources,
        "cost_per_record": cost_per_record,
        "total_cost": total_cost,
        "verified_records": verified_records,
        "unverified_records": unverified_records,
        "assignments": rule_assignments,
    }

In [89]:
source_cost_2 = {"a": 10, "b": 5, "c": 7, "d": 3, 
                 "e": 11, "f": 12, "g": 13, "h": 14}

In [90]:
output = ds_selection_optimizer(df_verified, source_cost_2, min_verified_rate=0.8, time_limit_sec=60)


In [98]:
df_verified

,record_id,rule_results
0,record_1,"{'rule_1': ['b', 'c'], 'rule_2': ['b', 'c', 'f..."
1,record_10,"{'rule_1': ['a', 'd'], 'rule_2': ['d', 'f']}"
2,record_100,"{'rule_1': ['a', 'b', 'd', 'e', 'f'], 'rule_2'..."
3,record_1000,"{'rule_1': ['a'], 'rule_2': ['c']}"
4,record_101,"{'rule_1': ['g'], 'rule_2': ['c', 'd', 'e', 'h']}"
...,...,...
494,record_543,"{'rule_1': ['a', 'b'], 'rule_2': ['a', 'b']}"
495,record_544,"{'rule_1': ['a', 'b'], 'rule_2': ['a', 'b']}"
496,record_545,"{'rule_1': ['a'], 'rule_2': ['a', 'b', 'd']}"
497,record_546,"{'rule_1': ['b'], 'rule_2': ['d']}"


In [ ]:
import math
import pulp


def solve_source_selection_pulp(records, source_cost, min_verify_rate=1.0, time_limit_sec=60):
    sources = sorted(source_cost.keys())
    n_records = len(records)
    required_verified = math.ceil(min_verify_rate * n_records)

    model = pulp.LpProblem("source_selection", pulp.LpMinimize)

    x = {
        d: pulp.LpVariable(f"select_{d}", cat="Binary")
        for d in sources
    }

    z = {
        r: pulp.LpVariable(f"record_{r}_verified", cat="Binary")
        for r in range(n_records)
    }

    y = {}

    for r, record_rules in enumerate(records):
        rules = list(record_rules.keys())

        for rule in rules:
            valid_candidates = [
                d for d in record_rules[rule]
                if d in source_cost
            ]

            rule_assignments = []

            for d in valid_candidates:
                var = pulp.LpVariable(f"assign_r{r}_{rule}_{d}", cat="Binary")
                y[(r, rule, d)] = var
                rule_assignments.append(var)

                model += var <= x[d]

            model += pulp.lpSum(rule_assignments) == z[r]

        for d in sources:
            assignments_using_d = [
                y[(r, rule, d)]
                for rule in rules
                if (r, rule, d) in y
            ]
            if assignments_using_d:
                model += pulp.lpSum(assignments_using_d) <= 1

    model += pulp.lpSum(z[r] for r in range(n_records)) >= required_verified

    model += pulp.lpSum(source_cost[d] * x[d] for d in sources)

    solver = pulp.PULP_CBC_CMD(timeLimit=time_limit_sec, msg=False)
    status_code = model.solve(solver)

    status = pulp.LpStatus[status_code]

    if status not in ("Optimal", "Feasible"):
        return {
            "status": "NO_SOLUTION",
            "min_verify_rate": min_verify_rate,
        }

    selected_sources = [
        d for d in sources
        if pulp.value(x[d]) > 0.5
    ]

    verified_records = [
        r for r in range(n_records)
        if pulp.value(z[r]) > 0.5
    ]

    unverified_records = [
        r for r in range(n_records)
        if pulp.value(z[r]) < 0.5
    ]

    assignments = {}

    for r in verified_records:
        assignments[r] = {}
        for rule in records[r]:
            for d in records[r][rule]:
                if (r, rule, d) in y and pulp.value(y[(r, rule, d)]) > 0.5:
                    assignments[r][rule] = d

    cost_per_record = sum(source_cost[d] for d in selected_sources)
    total_cost = cost_per_record * n_records

    return {
        "status": status,
        "min_verify_rate": min_verify_rate,
        "required_verified": required_verified,
        "actual_verified": len(verified_records),
        "actual_verify_rate": len(verified_records) / n_records,
        "selected_sources": selected_sources,
        "cost_per_record": cost_per_record,
        "total_cost": total_cost,
        "verified_records": verified_records,
        "unverified_records": unverified_records,
        "assignments": assignments,
    }

In [96]:
output = solve_source_selection_pulp(list(df_verified['rule_results']), source_cost_2, min_verify_rate=0.8, time_limit_sec=60)

In [97]:
output

{'status': 'Optimal',
 'min_verify_rate': 0.8,
 'required_verified': 356,
 'actual_verified': 381,
 'actual_verify_rate': 0.8561797752808988,
 'selected_sources': ['a', 'b', 'c', 'd'],
 'cost_per_record': 25,
 'total_cost': 11125,
 'verified_records': [0,
  1,
  2,
  3,
  5,
  6,
  7,
  8,
  9,
  10,
  12,
  13,
  14,
  16,
  17,
  18,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  36,
  37,
  38,
  39,
  41,
  42,
  43,
  44,
  45,
  47,
  48,
  49,
  50,
  51,
  52,
  53,
  55,
  56,
  57,
  58,
  59,
  60,
  61,
  62,
  63,
  65,
  66,
  68,
  69,
  70,
  72,
  73,
  74,
  75,
  76,
  77,
  78,
  79,
  81,
  82,
  83,
  84,
  85,
  86,
  87,
  88,
  89,
  90,
  91,
  94,
  95,
  96,
  97,
  98,
  99,
  100,
  101,
  102,
  103,
  104,
  105,
  106,
  107,
  108,
  109,
  111,
  112,
  113,
  114,
  115,
  116,
  117,
  118,
  119,
  121,
  122,
  123,
  124,
  125,
  126,
  127,
  128,
  129,
  130,
  131,
  132,
  133,
  134,
  135,
  13

In [ ]:
import math
import pandas as pd
import pulp


def solve_source_selection_pulp(
        df, source_cost,
        min_verify_rate=1.0, time_limit_sec=60,
    ):


    required_cols = {"record_id", "rule_results"}
    missing = required_cols - set(df.columns)

    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = df.reset_index(drop=True).copy()

    sources = sorted(source_cost.keys())
    n_records = len(df)

    required_verified = math.ceil(min_verify_rate * n_records)
    model = pulp.LpProblem("source_selection", pulp.LpMinimize)


    # Data source selection variables
    x = {d: pulp.LpVariable(f"select_{d}", cat="Binary") for d in sources}

    # Record verification variables
    z = {r: pulp.LpVariable(f"record_{r}_verified", cat="Binary") for r in range(n_records)}

    # Rule assignment variables: y[(r, rule, d)] = 1 if record r's rule is satisfied by data source d
    y = {}


    for r in range(n_records):

        record_rules = df.loc[r, "rule_results"]
        rules = list(record_rules.keys())

        for rule in rules:
            valid_candidates = [d for d in record_rules[rule] if d in source_cost]
            rule_assignments = []

            for d in valid_candidates:
                assign_var = pulp.LpVariable(f"assign_r{r}_{rule}_{d}", cat="Binary")
                y[(r, rule, d)] = assign_var
                rule_assignments.append(assign_var)

                model += assign_var <= x[d]

            model += (pulp.lpSum(rule_assignments)== z[r])

        for d in sources:
            assignments_using_d = [y[(r, rule, d)] for rule in rules if (r, rule, d) in y]
            if assignments_using_d:
                model += ( pulp.lpSum(assignments_using_d) <= 1)


    model += (pulp.lpSum(z[r] for r in range(n_records)) >= required_verified)

    # --------------------------------------------------
    # Objective
    # --------------------------------------------------

    model += pulp.lpSum(source_cost[d] * x[d] for d in sources)

    # --------------------------------------------------
    # Solve
    # --------------------------------------------------

    solver = pulp.PULP_CBC_CMD(timeLimit=time_limit_sec,msg=False)

    status_code = model.solve(solver)

    status = pulp.LpStatus[status_code]

    if status != "Optimal":

        return {
            "status": "NO_SOLUTION",
            "solver_status": status,
            "min_verify_rate": min_verify_rate,
        }

    # --------------------------------------------------
    # Extract results
    # --------------------------------------------------

    selected_sources = [d for d in sources if pulp.value(x[d]) > 0.5]
    verified_rows = [r for r in range(n_records) if pulp.value(z[r]) > 0.5]
    unverified_rows = [r for r in range(n_records) if pulp.value(z[r]) < 0.5]
    verified_record_ids = (df.loc[verified_rows, "record_id"].tolist())
    unverified_record_ids = (df.loc[unverified_rows, "record_id"].tolist())

    assignments = {}

    for r in verified_rows:

        record_id = df.loc[r, "record_id"]
        record_rules = df.loc[r, "rule_results"]
        assignments[record_id] = {}

        for rule in record_rules:
            for d in record_rules[rule]:
                if ((r, rule, d) in y and pulp.value(y[(r, rule, d)]) > 0.5):
                    assignments[record_id][rule] = d

    cost_per_record = sum(source_cost[d] for d in selected_sources)

    total_cost = cost_per_record * n_records

    return {
        "status": status,
        "min_verify_rate": min_verify_rate,
        "required_verified": required_verified,
        "actual_verified": len(verified_record_ids),
        "actual_verify_rate": (len(verified_record_ids) / n_records),
        "selected_sources": selected_sources,
        "cost_per_record": cost_per_record,
        "total_cost": total_cost,
        "verified_record_ids": verified_record_ids,
        "unverified_record_ids": unverified_record_ids,
        "assignments": assignments,
    }

In [104]:
output = solve_source_selection_pulp(
    df_verified,
    source_cost_2,
    min_verify_rate=0.8,
    time_limit_sec=60,
)

In [105]:
output

{'status': 'Optimal',
 'min_verify_rate': 0.8,
 'required_verified': 356,
 'actual_verified': 381,
 'actual_verify_rate': 0.8561797752808988,
 'selected_sources': ['a', 'b', 'c', 'd'],
 'cost_per_record': 25,
 'total_cost': 11125,
 'verified_record_ids': ['record_1',
  'record_10',
  'record_100',
  'record_1000',
  'record_103',
  'record_104',
  'record_105',
  'record_106',
  'record_107',
  'record_109',
  'record_110',
  'record_112',
  'record_113',
  'record_116',
  'record_117',
  'record_118',
  'record_12',
  'record_120',
  'record_121',
  'record_122',
  'record_123',
  'record_124',
  'record_125',
  'record_127',
  'record_128',
  'record_13',
  'record_131',
  'record_132',
  'record_133',
  'record_135',
  'record_136',
  'record_138',
  'record_139',
  'record_14',
  'record_140',
  'record_142',
  'record_143',
  'record_144',
  'record_145',
  'record_146',
  'record_148',
  'record_149',
  'record_15',
  'record_151',
  'record_152',
  'record_153',
  'record_154',
